# Improved Disease Prediction Model Training

This notebook contains the improved architecture and preprocessing logic for the disease prediction service.

In [1]:
import pandas as pd
import numpy as np
import pickle
import json
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, Input, Add, Activation
from tensorflow.keras.utils import to_categorical

/Users/ranitmanik/code/SRSA/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


### 1. Load and Clean Data

In [2]:
print("Loading dataset...")
df = pd.read_csv("../data/dataset.csv", low_memory=False)
X = df.drop("diseases", axis=1)
y = df["diseases"]

# Remove zero-variance features (symptoms never present in the data)
variances = X.var()
zero_var_cols = variances[variances == 0].index.tolist()
print(f"Removing {len(zero_var_cols)} zero-variance features.")
X = X.drop(columns=zero_var_cols)

# Label Encoding for Target
le = LabelEncoder()
y_encoded = le.fit_transform(y)
num_classes = len(le.classes_)

# Split (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)

# Calculate Class Weights to handle extreme imbalance
class_weights = compute_class_weight('balanced', classes=np.unique(y_encoded), y=y_encoded)
class_weight_dict = dict(enumerate(class_weights))

# One-hot encoding for ANN training
y_train_cat = to_categorical(y_train, num_classes=num_classes)
y_test_cat = to_categorical(y_test, num_classes=num_classes)

Loading dataset...
Removing 49 zero-variance features.


### 2. Build Improved ANN Architecture

We use **Residual Blocks** to improve gradient flow and **Label Smoothing** to handle ambiguous symptom cases.

In [3]:
def build_improved_ann(input_dim, output_dim):
    inputs = Input(shape=(input_dim,))
    
    x = Dense(512, activation='relu')(inputs)
    x = BatchNormalization()(x)
    x = Dropout(0.4)(x)
    
    # Residual Block 1
    res = x
    x = Dense(512, activation='relu')(x)
    x = BatchNormalization()(x)
    x = Add()([x, res])
    x = Activation('relu')(x)
    x = Dropout(0.4)(x)
    
    x = Dense(256, activation='relu')(x)
    x = BatchNormalization()(x)
    
    # Residual Block 2
    res2 = x
    x = Dense(256, activation='relu')(x)
    x = BatchNormalization()(x)
    x = Add()([x, res2])
    x = Activation('relu')(x)
    x = Dropout(0.3)(x)
    
    x = Dense(128, activation='relu')(x)
    x = BatchNormalization()(x)
    
    outputs = Dense(output_dim, activation='softmax')(x)
    
    model = Model(inputs=inputs, outputs=outputs)
    
    # Label Smoothing (0.1) handles ambiguous symptom vectors
    loss = tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1)
    model.compile(optimizer='adam', loss=loss, metrics=['accuracy'])
    return model

### 3. Training

In [4]:
EPOCHS = 50
print(f"Training Improved ANN for {EPOCHS} epochs...")
model = build_improved_ann(X_train.shape[1], num_classes)

history = model.fit(
    X_train, y_train_cat, 
    validation_data=(X_test, y_test_cat),
    epochs=EPOCHS, 
    batch_size=128, 
    class_weight=class_weight_dict,
    verbose=1
)

Training Improved ANN for 50 epochs...
Epoch 1/50
1544/1544 ━━━━━━━━━━━━━━━━━━━━ 10s 5ms/step - accuracy: 0.2069 - loss: 5.3824 - val_accuracy: 0.7630 - val_loss: 2.0401
Epoch 2/50
1544/1544 ━━━━━━━━━━━━━━━━━━━━ 7s 5ms/step - accuracy: 0.6512 - loss: 2.5905 - val_accuracy: 0.8203 - val_loss: 1.8496
Epoch 3/50
1544/1544 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.7106 - loss: 2.2338 - val_accuracy: 0.8251 - val_loss: 1.8098
Epoch 4/50
1544/1544 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - accuracy: 0.7279 - loss: 2.1294 - val_accuracy: 0.8299 - val_loss: 1.8054
Epoch 5/50
1544/1544 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - accuracy: 0.7433 - loss: 2.0678 - val_accuracy: 0.8359 - val_loss: 1.7815
Epoch 6/50
1544/1544 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - accuracy: 0.7575 - loss: 1.9263 - val_accuracy: 0.8408 - val_loss: 1.7740
Epoch 7/50
1544/1544 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - accuracy: 0.7681 - loss: 1.8939 - val_accuracy: 0.8400 - val_loss: 1.7198
Epoch 8/50
1544/1544 ━━━━━━━━━━━━━━━━━━━━ 7s 5ms/s

### 4. Save Artifacts for API

In [5]:
artifact_dir = "../models/improved"
os.makedirs(artifact_dir, exist_ok=True)

print(f"Saving artifacts to {artifact_dir}...")
model.save(f"{artifact_dir}/model.h5")

with open(f"{artifact_dir}/dropped_features.json", "w") as f:
    json.dump(zero_var_cols, f)

with open(f"{artifact_dir}/label_encoder.pkl", "wb") as f:
    pickle.dump(le, f)

with open(f"{artifact_dir}/features.json", "w") as f:
    json.dump(X.columns.tolist(), f)

print("Training Complete. The service is now ready to use the retrained model.")

Saving artifacts to ../models/improved...
Training Complete. The service is now ready to use the retrained model.
